# Wire a LangChain LCEL chain (retriever, prompt, LLM, parser), trace it with MLflow, and prove three questions answer grounded in the Lab 5 index

Lab 4 produced chunks. Lab 5 produced a Vector Search index over those chunks. Your PM wants a working chat answer to questions like "how do I sign up for Free Edition" in this session. You have everything you need; you have not composed it yet. The exam tests both the LCEL pipe shape and the MLflow tracing surface that makes the chain debuggable when it goes wrong.

The hands-on work:

- Stand up a fresh lab schema (`workspace.default.labrun_rag_chain`), a `test_queries` Delta table with three fixture questions, and a `chain_runs` Delta table to persist every invocation.
- Instantiate a `DatabricksVectorSearch` retriever against the Lab 5 endpoint and index, plus a `ChatDatabricks` LLM bound to `databricks-meta-llama-3-3-70b-instruct`.
- Compose the LCEL chain `{context, question} | prompt | llm | parser`, with a `RunnableLambda(format_docs)` step joining the retriever's Document list into a single context string.
- Invoke the chain three times inside `mlflow.start_run()` with `mlflow.langchain.autolog()` on, persist each run to `chain_runs`, and confirm every run shows a retriever span and an LLM span in its trace.
- Add a `RunnableBranch` intent router that picks between a how-to prompt and a pricing prompt based on a keyword heuristic.

**Time.** Plan for about 75 minutes hands-on. The Lab 5 endpoint cold-start can add 10 to 30 seconds on the first retrieval. Budget up to 120 minutes for the session window.

**Cost.** Between $0.00 and $0.05 per session. Three chain invocations cost a couple of cents in Foundation Model API tokens. Three retrievals cost fractions of a cent. MLflow tracing is free on Databricks-hosted MLflow. If you re-run the chain twenty times debugging, the session total tops out around five cents. The MLflow trace UI is where you should spend your time, not staring at token counts.

This lab maps to Databricks GenAI Engineer Associate Domain 3 (Application Development, ~30% provisional) and Domain 4 (Assembling and Deploying Applications, ~22% provisional).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the chain dependencies. Pinned per LAB-CREATION-BLUEPRINT.
# Never use unpinned installs. The databricks-langchain package supplies
# DatabricksVectorSearch and ChatDatabricks; mlflow gives us langchain autolog.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 langchain==0.3.7 langchain-community==0.3.5 databricks-langchain==0.2.0 mlflow==2.17.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers use snake_case under the
# labrun_ prefix because UC prefers snake_case (hyphens force backtick-quoting
# everywhere downstream). Endpoint and index identifiers inherited from Lab 5.

import atexit
import getpass
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from databricks.vector_search.client import VectorSearchClient

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableBranch, RunnableLambda, RunnablePassthrough
import mlflow
from mlflow.tracking import MlflowClient

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "rag-chain-with-langchain-and-foundation-models"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[5].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_rag_chain"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
TEST_QUERIES_FQN = f"{LAB_SCHEMA_FQN}.test_queries"
CHAIN_RUNS_FQN = f"{LAB_SCHEMA_FQN}.chain_runs"

# Inherited from Lab 5. The setup cell verifies both are alive before any task runs.
ENDPOINT_NAME = "labrun-vs-endpoint"
INDEX_NAME = "workspace.default.labrun_genai_corpus.chunks_idx"

# Foundation Model API LLM endpoint. Pay-per-token. Free Edition default.
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# Three fixture queries the chain runs against. One howto, one pricing, one generic.
# Intent labels and expected-keyword sets are used by Checkpoint 4 to assert that
# the RunnableBranch routed the howto query through the howto template and the
# pricing query through the pricing template.
TEST_QUERIES = [
    (1, "how do I sign up for Databricks Free Edition", "howto", "step-by-step"),
    (2, "how much does Databricks Free Edition cost", "pricing", "cost"),
    (3, "what is Unity Catalog", "default", ""),
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and confirm the Lab 5 endpoint + index are alive
# before any task runs. Enable mlflow.langchain.autolog so every chain
# invocation inside mlflow.start_run() captures a trace.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

import os
os.environ["DATABRICKS_HOST"] = databricks_host
os.environ["DATABRICKS_TOKEN"] = databricks_token

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
MLFLOW_EXPERIMENT_PATH = f"/Users/{CALLER_USER}/labrun-rag-chain"
print(f"Credentials validated. Workspace user: {CALLER_USER}")
print(f"MLflow experiment target: {MLFLOW_EXPERIMENT_PATH}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
    "caller_user": CALLER_USER,
    "mlflow_experiment_path": MLFLOW_EXPERIMENT_PATH,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


# Verify the Lab 5 endpoint and index are alive. If the Lab 5 cleanup tore
# them down, Lab 6 cannot proceed; re-run Lab 5 to stand them up again.
vsc = VectorSearchClient(workspace_url=databricks_host, personal_access_token=databricks_token, disable_notice=True)

try:
    endpoint_info = vsc.get_endpoint(name=ENDPOINT_NAME)
except Exception as exc:
    print(f"Lab 5 Vector Search endpoint {ENDPOINT_NAME!r} is not reachable: {exc}")
    print()
    print("Re-run Lab 5 (mosaic-ai-vector-search-delta-sync-index) to recreate")
    print("the endpoint and the index before continuing with Lab 6.")
    raise SystemExit(1)

try:
    index_handle = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    _ = index_handle.describe()
except Exception as exc:
    print(f"Lab 5 Vector Search index {INDEX_NAME!r} is not reachable: {exc}")
    print()
    print("Re-run Lab 5 to recreate the Delta-Sync index over the corpus table.")
    raise SystemExit(1)

print(f"Lab 5 endpoint alive: {ENDPOINT_NAME}")
print(f"Lab 5 index alive: {INDEX_NAME}")

# Configure MLflow once. set_experiment will create the experiment on first run.
mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)
mlflow.langchain.autolog()
print(f"MLflow autolog enabled for experiment: {MLFLOW_EXPERIMENT_PATH}")

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Manifest is module-
# level and in reverse-creation order. The MLflow experiment soft-deletes
# (Databricks-hosted MLflow moves it to trash; auto-purge handles the rest).
# The Lab 5 endpoint and index are NOT in this manifest; they survive into
# Labs 7 through 9 per the cert YAML.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="mlflow_experiment",
        id=MLFLOW_EXPERIMENT_PATH,
        region="databricks",
        cli_delete_command=(
            "databricks experiments delete-experiment "
            f"$(databricks experiments get-by-name --experiment-name {MLFLOW_EXPERIMENT_PATH} "
            "--output JSON | jq -r .experiment.experiment_id)"
        ),
    ),
    CleanupResource(
        type="uc_managed_table",
        id=CHAIN_RUNS_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHAIN_RUNS_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=TEST_QUERIES_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {TEST_QUERIES_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against Databricks Unity Catalog (via the SQL Statement Execution API) and
    Databricks-hosted MLflow (via mlflow.tracking.MlflowClient). Supports the
    GenAI track resource types this lab creates: mlflow_experiment,
    uc_managed_table, uc_schema."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "mlflow_experiment":
            client = MlflowClient()
            try:
                existing = client.get_experiment_by_name(rid)
            except Exception:
                existing = None
            if existing is None:
                return
            try:
                client.delete_experiment(existing.experiment_id)
            except Exception as exc:
                msg = str(exc).lower()
                if "not found" in msg or "does not exist" in msg or "resource_does_not_exist" in msg:
                    return
                raise
        elif rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "mlflow_experiment":
            client = MlflowClient()
            try:
                existing = client.get_experiment_by_name(rid)
            except Exception:
                return False
            if existing is None:
                return False
            # Soft-deleted experiments report lifecycle_stage 'deleted' but
            # remain visible until purge. Treat soft-deleted as gone.
            stage = getattr(existing, "lifecycle_stage", "active")
            return stage == "active"
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return list of Unity Catalog identifiers tagged with the lab slug.

        MLflow experiments are not surfaced through information_schema, so the
        adapter checks the named experiment path directly and reports it if it
        is active and not in the current manifest."""
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        # MLflow experiment check: report the path if it exists and is active.
        try:
            existing = MlflowClient().get_experiment_by_name(credentials.get("mlflow_experiment_path", MLFLOW_EXPERIMENT_PATH))
            if existing is not None and getattr(existing, "lifecycle_stage", "active") == "active":
                arns.append(credentials.get("mlflow_experiment_path", MLFLOW_EXPERIMENT_PATH))
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
# Filter: anything still in the manifest is expected on a re-run; the orphan
# scan only blocks on objects from PRIOR sessions that the manifest did not
# register. The first run of the lab in a fresh workspace has zero orphans.
manifest_ids = {r.id for r in CLEANUP_MANIFEST}
true_orphans = [o for o in orphans if o not in manifest_ids]

if true_orphans:
    print(f"BLOCKED: existing labrun objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in true_orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the")
    print("cleanup cell at the bottom of this notebook first, or drop them")
    print("manually via the SQL commands the cleanup cell prints, then")
    print("re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the lab schema and tables, instantiate the retriever and LLM, sanity-check both

The chain has four moving parts: the retriever (talks to the Lab 5 Vector Search index), the prompt template (interpolates the retrieved context and the user question into a single string), the LLM (the pay-per-token Foundation Model API endpoint), and the output parser (coerces the LLM response to a plain string). Two of them connect to live cloud services. Build those first and confirm they work in isolation before composing.

Build it in this order:

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_rag_chain` and tag it with `labrun_lab_slug = rag-chain-with-langchain-and-foundation-models`.
2. Create `test_queries` (`query_id INT`, `query STRING`, `intent_label STRING`, `expected_keyword STRING`) and seed it from the `TEST_QUERIES` constant in scope. Tag the table.
3. Create `chain_runs` (`run_id INT`, `query STRING`, `retrieved_chunk_ids ARRAY<STRING>`, `llm_response STRING`, `mlflow_run_id STRING`, `latency_ms BIGINT`, `recorded_at TIMESTAMP`). Tag the table.
4. Instantiate `retriever = DatabricksVectorSearch(endpoint=ENDPOINT_NAME, index_name=INDEX_NAME, columns=['chunk_id','chunk_text']).as_retriever(search_kwargs={'k': 3})`.
5. Instantiate `llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)`.
6. Sanity test: call `retriever.invoke(TEST_QUERIES[0][1])` and confirm it returns 3 Document objects; call `llm.invoke("reply with READY")` and confirm a non-empty response.

Checkpoint 1 re-instantiates the retriever and LLM in the validator's own scope (no shared state) and runs the same sanity probes.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the schema, the two Delta tables, and instantiate the
# retriever + LLM. The CREATEs are idempotent. The retriever and LLM live in
# module scope so subsequent tasks can compose them into the chain.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Apply the lab tag to the schema with ALTER SCHEMA ... SET TAGS.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for TEST_QUERIES_FQN with columns
# (query_id INT, query STRING, intent_label STRING, expected_keyword STRING)
# USING DELTA.

# YOUR CODE: Apply the lab tag to TEST_QUERIES_FQN.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS for CHAIN_RUNS_FQN with columns
# (run_id INT, query STRING, retrieved_chunk_ids ARRAY<STRING>,
#  llm_response STRING, mlflow_run_id STRING, latency_ms BIGINT,
#  recorded_at TIMESTAMP) USING DELTA.

# YOUR CODE: Apply the lab tag to CHAIN_RUNS_FQN.

# Seed test_queries from the TEST_QUERIES constant. Idempotent: TRUNCATE then
# INSERT so a re-run produces the same fixture rows.
run_sql(f"TRUNCATE TABLE {TEST_QUERIES_FQN}")
values = ", ".join(
    f"({qid}, '{q}', '{intent}', '{kw}')"
    for (qid, q, intent, kw) in TEST_QUERIES
)
run_sql(
    f"INSERT INTO {TEST_QUERIES_FQN} (query_id, query, intent_label, expected_keyword) VALUES {values}"
)

# YOUR CODE: Instantiate the retriever as a module-level variable named
# `retriever` with DatabricksVectorSearch(endpoint=ENDPOINT_NAME,
# index_name=INDEX_NAME, columns=['chunk_id','chunk_text']).as_retriever(
# search_kwargs={'k': 3}).

# YOUR CODE: Instantiate the LLM as a module-level variable named `llm` with
# ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0).

# Sanity test the two components in isolation.
sample_docs = retriever.invoke(TEST_QUERIES[0][1])
print(f"Retriever returned {len(sample_docs)} documents for the fixture query.")
print(f"  First chunk preview: {sample_docs[0].page_content[:120]!r}")

sample_response = llm.invoke("reply with READY")
preview = sample_response.content if hasattr(sample_response, "content") else str(sample_response)
print(f"LLM responded: {preview[:80]!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: in the validator's own scope, DatabricksVectorSearch returns 3
# chunks for a fixture query and ChatDatabricks returns a non-empty response
# to a one-token completion. Independent of module-level state.


def checkpoint_1(session):
    try:
        # Verify the schema and both tables exist.
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {LAB_SCHEMA_FQN} not found. Did CREATE SCHEMA run?"
                ),
            )
        for table_fqn, table_name in (
            (TEST_QUERIES_FQN, "test_queries"),
            (CHAIN_RUNS_FQN, "chain_runs"),
        ):
            table_sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
                f"AND table_name = '{table_name}'"
            )
            if not run_sql(table_sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Table {table_fqn} not found. Re-run Task 1 to create it."
                    ),
                )
        count_result = run_sql(f"SELECT count(*) FROM {TEST_QUERIES_FQN}")
        seeded = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if seeded < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{TEST_QUERIES_FQN} has {seeded} rows; expected at least 3 "
                    f"fixture queries. Re-seed from TEST_QUERIES."
                ),
            )

        # Independent retriever instantiation. No reliance on module-level `retriever`.
        validator_retriever = DatabricksVectorSearch(
            endpoint=ENDPOINT_NAME,
            index_name=INDEX_NAME,
            columns=["chunk_id", "chunk_text"],
        ).as_retriever(search_kwargs={"k": 3})
        docs = validator_retriever.invoke(TEST_QUERIES[0][1])
        if not isinstance(docs, list) or len(docs) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Retriever returned {len(docs) if isinstance(docs, list) else 'non-list'} "
                    f"chunks; expected at least 3. Confirm Lab 5's chunks_idx is populated "
                    f"and the endpoint is awake (cold-start can take 10 to 30 seconds)."
                ),
            )

        # Independent LLM instantiation and one-token probe.
        validator_llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)
        probe = validator_llm.invoke("reply with READY")
        content = probe.content if hasattr(probe, "content") else str(probe)
        if not content or not content.strip():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ChatDatabricks endpoint {LLM_ENDPOINT} returned an empty response. "
                    f"Check that the pay-per-token endpoint is enabled in this workspace."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message points at the exact missing piece: a schema row, a table row, the seed count, the retriever's chunk count, or the LLM's empty response. Each branch suggests a different fix. The retriever path can fail with zero chunks if the Lab 5 endpoint is asleep; the cure is to wait a few seconds and re-run the cell.

</details>

<details><summary>Hint 2 (direction)</summary>

Five things to write. The two CREATE TABLE statements use `(col_name TYPE, ...) USING DELTA`. The retriever pattern is `DatabricksVectorSearch(endpoint=..., index_name=..., columns=[...]).as_retriever(search_kwargs={'k': 3})`. The LLM pattern is `ChatDatabricks(endpoint=..., temperature=0.0)`. The schema and table tags both use `ALTER ... SET TAGS ('labrun_lab_slug' = '...')`. The constants `LAB_SCHEMA_FQN`, `TEST_QUERIES_FQN`, `CHAIN_RUNS_FQN`, `ENDPOINT_NAME`, `INDEX_NAME`, `LLM_ENDPOINT`, `LAB_TAG_KEY`, `LAB_TAG_VALUE` are already in scope.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(
    f"ALTER SCHEMA {LAB_SCHEMA_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"CREATE TABLE IF NOT EXISTS {TEST_QUERIES_FQN} ("
    f"  query_id INT, query STRING, intent_label STRING, expected_keyword STRING"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {TEST_QUERIES_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
run_sql(
    f"CREATE TABLE IF NOT EXISTS {CHAIN_RUNS_FQN} ("
    f"  run_id INT, query STRING, retrieved_chunk_ids ARRAY<STRING>, "
    f"  llm_response STRING, mlflow_run_id STRING, latency_ms BIGINT, "
    f"  recorded_at TIMESTAMP"
    f") USING DELTA"
)
run_sql(
    f"ALTER TABLE {CHAIN_RUNS_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

retriever = DatabricksVectorSearch(
    endpoint=ENDPOINT_NAME,
    index_name=INDEX_NAME,
    columns=["chunk_id", "chunk_text"],
).as_retriever(search_kwargs={"k": 3})

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)
```

</details>

**Wallet check.** One retrieval (3 chunks, fractions of a cent), one one-token LLM probe (well under a cent). Total damage so far: under one cent. Your morning coffee cost roughly 400x more. Open MLflow > Experiments > labrun-rag-chain in another tab; the next task fills it with traces.

## Task 2: Compose the LCEL chain and run it against the three fixture queries

LCEL composes Runnables with the pipe operator. The contract is type-strict: the upstream component's output type must match the downstream component's input type, and the pipe does NOT auto-coerce. The retriever emits a `List[Document]`. The prompt template expects a string for its `{context}` placeholder. A `RunnableLambda(format_docs)` joins the documents into one string, which is the canonical pattern the exam tests directly (Domain 4: "Code a simple chain according to requirements").

The shape:

```
{"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
  | prompt
  | llm
  | StrOutputParser()
```

Then loop over the three fixture queries. Each invocation runs inside `mlflow.start_run()` so the autolog enabled in the setup cell captures a trace per run. Persist the chain output and the `mlflow_run_id` to `chain_runs` so the next checkpoint can correlate runs to traces.

Assign the composed chain to a module-level variable named `rag_chain`. The checkpoint invokes it directly.

In [ ]:
# NBVAL_SKIP
# Task 2: compose the LCEL chain, loop over TEST_QUERIES, persist each run.
# Every invocation lives inside mlflow.start_run() so the autolog captures a
# trace; the trace ID is stored alongside the response in chain_runs.

def format_docs(docs):
    """Join the retriever's Document list into a single string for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


# YOUR CODE: Build a PromptTemplate from the template string
#   "Answer the question using only the context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
# and assign it to `prompt`.

# YOUR CODE: Compose the chain via the LCEL pipe operator. The shape:
#   rag_chain = (
#       {"context": retriever | RunnableLambda(format_docs),
#        "question": RunnablePassthrough()}
#       | prompt | llm | StrOutputParser()
#   )
# Assign the result to a module-level variable named `rag_chain`.


# Drive the three queries through the chain. Each run lives inside an MLflow
# run so autolog captures the trace. Persist the response and the run_id.
run_sql(f"TRUNCATE TABLE {CHAIN_RUNS_FQN}")
persist_rows = []
for (qid, q, intent, _kw) in TEST_QUERIES:
    with mlflow.start_run(run_name=f"task2_q{qid}") as run:
        t0 = time.time()
        response = rag_chain.invoke(q)
        latency_ms = int((time.time() - t0) * 1000)
        retrieved_docs = retriever.invoke(q)
        chunk_ids = [
            str(d.metadata.get("chunk_id", ""))
            for d in retrieved_docs
        ]
        chunk_ids_sql = "array(" + ", ".join(f"'{cid}'" for cid in chunk_ids) + ")"
        response_sql_safe = response.replace("'", "''")[:4000]
        persist_rows.append((qid, q, chunk_ids_sql, response_sql_safe, run.info.run_id, latency_ms))
        print(f"Q{qid} ({intent}): {response[:140]!r}")

for (qid, q, chunk_ids_sql, response_safe, run_id, latency_ms) in persist_rows:
    q_safe = q.replace("'", "''")
    run_sql(
        f"INSERT INTO {CHAIN_RUNS_FQN} "
        f"(run_id, query, retrieved_chunk_ids, llm_response, mlflow_run_id, latency_ms, recorded_at) "
        f"VALUES ({qid}, '{q_safe}', {chunk_ids_sql}, '{response_safe}', '{run_id}', {latency_ms}, current_timestamp())"
    )

print(f"\nPersisted {len(persist_rows)} chain runs to {CHAIN_RUNS_FQN}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: rag_chain exists in module scope, the validator invokes it
# against three fixture queries (independent of Task 2's loop), and chain_runs
# has at least three rows persisted.


def checkpoint_2(session):
    try:
        chain = globals().get("rag_chain")
        if chain is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Module-level variable `rag_chain` is not defined. Compose "
                    "the LCEL chain and assign it to `rag_chain` before re-running."
                ),
            )
        fixture_result = run_sql(
            f"SELECT query_id, query FROM {TEST_QUERIES_FQN} ORDER BY query_id LIMIT 3"
        )
        if len(fixture_result["rows"]) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{TEST_QUERIES_FQN} has fewer than 3 fixture queries. Re-run Task 1 to seed."
                ),
            )
        for row in fixture_result["rows"]:
            q = str(row[1])
            response = chain.invoke(q)
            if not isinstance(response, str) or not response.strip():
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"rag_chain.invoke({q!r}) returned an empty or non-string "
                        f"response. The chain must end in StrOutputParser() so the "
                        f"final output is a plain string."
                    ),
                )
        # Confirm chain_runs has at least three persisted rows.
        count_result = run_sql(f"SELECT count(*) FROM {CHAIN_RUNS_FQN}")
        rows_present = int(count_result["rows"][0][0]) if count_result["rows"] else 0
        if rows_present < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{CHAIN_RUNS_FQN} has {rows_present} rows; expected at least 3. "
                    f"Persist one INSERT per fixture query from the Task 2 loop."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The error message names exactly one of: `rag_chain` is undefined, the chain returned an empty response, or `chain_runs` is short on rows. A non-string response usually means the chain stops at `llm` without `StrOutputParser()` at the tail. A `List[Document]` type error at invoke time means the `RunnableLambda(format_docs)` step got skipped.

</details>

<details><summary>Hint 2 (direction)</summary>

Two pieces. The prompt template is `PromptTemplate.from_template("Answer the question using only the context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:")`. The chain composition uses a parallel dict at the head (`{"context": ..., "question": ...}`) so the retriever's output flows into `{context}` and the user's question flows into `{question}` via `RunnablePassthrough()`. The retriever emits Documents, so the lambda joins them into one string before the dict passes downstream.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
prompt = PromptTemplate.from_template(
    "Answer the question using only the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)
```

</details>

**Wallet check.** Three chain invocations, three retrievals, three LLM completions, roughly 1500 to 2500 tokens total. At pay-per-token rates that lands around 1 to 2 cents. The validator's three extra invocations add the same again. Total damage so far: under 5 cents. Open the MLflow Experiments tab in the workspace UI and pick `labrun-rag-chain` to inspect a trace for each run.

## Task 3: Confirm MLflow autolog captured a retriever span and an LLM span per run

Autolog was enabled in the setup cell (`mlflow.langchain.autolog()`). Every chain invocation inside `mlflow.start_run()` produced a trace with one span per component the chain executed. The exam tests this surface directly (Domain 6: "Evaluate agent performance using MLflow GenAI scoring and tracing").

Open the MLflow UI for `labrun-rag-chain` in the workspace, pick one of the runs from Task 2, click the Traces tab, and inspect the span tree. You should see at least:

- A retriever span (the `DatabricksVectorSearch.as_retriever()` call) with the query as input and a list of Documents as output.
- An LLM span (the `ChatDatabricks` call) with the formatted prompt as input and the model response as output.

The task code cell below is verification only: it re-affirms autolog is enabled in case a kernel restart cleared the registration. The checkpoint then walks the trace spans server-side via `MlflowClient.search_runs` plus `mlflow.get_trace(run_id)` and asserts the retriever + LLM span pattern across all three Task 2 runs.

In [ ]:
# NBVAL_SKIP
# Task 3: re-affirm autolog is enabled and the experiment is set. Idempotent;
# safe on a kernel restart that cleared the registration done in the setup cell.

# YOUR CODE: Call mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH) to ensure the
# active experiment is the lab's experiment (the setup cell already did this,
# but a kernel restart would have cleared it).

# YOUR CODE: Call mlflow.langchain.autolog() to re-enable tracing inside
# mlflow.start_run() contexts.

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
if experiment is None:
    print(f"Experiment not found at {MLFLOW_EXPERIMENT_PATH}.")
    print("Re-run the setup cell, then re-run Task 2 to produce traces.")
else:
    runs = MlflowClient().search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["attributes.start_time DESC"],
        max_results=10,
    )
    print(f"Experiment {MLFLOW_EXPERIMENT_PATH} has {len(runs)} recent runs.")
    for r in runs[:5]:
        print(f"  run_id={r.info.run_id} status={r.info.status} name={r.data.tags.get('mlflow.runName','')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: the lab's MLflow experiment has at least 3 runs from Task 2,
# and each run's trace exposes at least one retriever-style span and one
# LLM-style span. Span types come from the LangChain autolog naming scheme.


def _span_kind(span):
    """Best-effort classification of a span as retriever / llm / other based
    on span_type, name, and attributes. LangChain autolog uses span_type
    values like 'RETRIEVER', 'CHAT_MODEL', 'LLM', 'CHAIN' depending on the
    component shape; the name often contains 'Retriever' or 'ChatDatabricks'."""
    span_type = (getattr(span, "span_type", "") or "").upper()
    name = (getattr(span, "name", "") or "").lower()
    if span_type == "RETRIEVER" or "retriever" in name or "vectorsearch" in name:
        return "retriever"
    if span_type in ("LLM", "CHAT_MODEL") or "chatdatabricks" in name or "llm" in name:
        return "llm"
    return "other"


def checkpoint_3(session):
    try:
        client = MlflowClient()
        experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
        if experiment is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"MLflow experiment {MLFLOW_EXPERIMENT_PATH} not found. "
                    f"Re-run the setup cell so mlflow.set_experiment fires, then re-run Task 2."
                ),
            )
        runs = client.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=["attributes.start_time DESC"],
            max_results=50,
        )
        if len(runs) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Experiment has {len(runs)} runs; expected at least 3. Re-run "
                    f"Task 2's loop so mlflow.start_run() fires once per fixture query."
                ),
            )
        runs_to_check = runs[:3]
        for r in runs_to_check:
            run_id = r.info.run_id
            try:
                trace = mlflow.get_trace(run_id)
            except Exception as exc:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"mlflow.get_trace({run_id}) raised: {exc}. "
                        f"This usually means mlflow.langchain.autolog() was not active when "
                        f"rag_chain.invoke() ran. Re-run Task 3's autolog call, then re-run Task 2."
                    ),
                )
            if trace is None or not getattr(trace, "data", None):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Run {run_id} has no trace data. Autolog must be enabled BEFORE the chain "
                        f"is invoked inside mlflow.start_run(). Re-run Task 3 then Task 2."
                    ),
                )
            spans = trace.data.spans or []
            kinds = {_span_kind(s) for s in spans}
            if "retriever" not in kinds:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Run {run_id} has no retriever span (saw {len(spans)} spans of kinds "
                        f"{sorted(kinds)}). The chain must include the DatabricksVectorSearch "
                        f"retriever inside the autolog-captured invoke path."
                    ),
                )
            if "llm" not in kinds:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Run {run_id} has no LLM span (saw {len(spans)} spans of kinds "
                        f"{sorted(kinds)}). The chain must include the ChatDatabricks LLM step "
                        f"inside the autolog-captured invoke path."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message points at the specific gap: not enough runs (Task 2's loop did not fire all three iterations), no trace on a run (autolog was not active when the chain ran), or a span is missing (the chain composition does not include that component, or the chain invocation ran outside `mlflow.start_run()`). Each maps to a different fix.

</details>

<details><summary>Hint 2 (direction)</summary>

Two lines do the work. `mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)` pins the active experiment so the runs land in the right place. `mlflow.langchain.autolog()` enables the LangChain integration so every Runnable invocation inside `mlflow.start_run()` emits a span. The setup cell already called both, but a kernel restart between setup and Task 3 wipes the registration; re-affirming is cheap. After the cell prints, re-run Task 2 if the trace count is short.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)
mlflow.langchain.autolog()
```

If Task 2 ran before autolog was enabled, re-run Task 2 after this cell. The traces accumulate on the experiment.

</details>

**Wallet check.** Replaying a trace in the MLflow UI does NOT re-invoke the chain (the spans are captured artifacts), so opening the UI to inspect runs is zero token spend. Cumulative session spend stays under 5 cents.

## Task 4: Add intent-classification routing via RunnableBranch

Section 3 of the exam guide tests prompt augmentation by intent. Two ways to route: a keyword heuristic (fast, free, brittle on ambiguous queries) or a small FM API call (slower, costs a few tokens, more reliable). This lab uses the keyword heuristic because the fixture queries are intentionally unambiguous; production systems usually combine both.

Build two flavor-specific prompt templates:

- `HOWTO_PROMPT`: "You are a how-to assistant. Use the context to give step-by-step instructions..."
- `PRICING_PROMPT`: "You are a pricing assistant. Use the context to answer about cost..."

Write a router function that inspects the question, returns the string `'howto'` if it starts with "how do", `'pricing'` if it mentions "cost" or "price" or starts with "how much", and `'default'` otherwise.

Compose `routed_chain` so the router decides which prompt branch fires, then the rest of the pipe (retriever, llm, parser) runs the same way it did in Task 2.

Run the chain against the howto fixture and the pricing fixture inside `mlflow.start_run()` so the traces are captured. The checkpoint walks the prompt span of each trace and asserts it contains the branch-specific substring (`step-by-step` for howto, `cost` for pricing).

In [ ]:
# NBVAL_SKIP
# Task 4: build the intent router and the RunnableBranch composition.
# Persist the two intent runs to chain_runs so they show up in the
# end-of-task wallet check, and so the checkpoint can correlate.

HOWTO_PROMPT = PromptTemplate.from_template(
    "You are a how-to assistant. Use the context to give step-by-step "
    "instructions answering the user's question.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Step-by-step answer:"
)

PRICING_PROMPT = PromptTemplate.from_template(
    "You are a pricing assistant. Use the context to answer the user's "
    "question about cost or price as precisely as possible.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Cost answer:"
)

DEFAULT_PROMPT = PromptTemplate.from_template(
    "Answer the question using only the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)


# YOUR CODE: Implement a router function `classify_intent(payload)` that
# takes the parallel-dict payload (a dict with keys 'context' and 'question')
# and returns one of 'howto', 'pricing', 'default'. Inspect payload['question'].
# Heuristic: lowercase the question, return 'howto' if it starts with 'how do',
# 'pricing' if it contains 'cost' or 'price' or starts with 'how much',
# 'default' otherwise.


# YOUR CODE: Compose `routed_chain` with a RunnableBranch step between the
# parallel-dict head and the LLM. The shape:
#   routed_chain = (
#       {"context": retriever | RunnableLambda(format_docs),
#        "question": RunnablePassthrough()}
#       | RunnableBranch(
#           (lambda p: classify_intent(p) == 'howto', HOWTO_PROMPT),
#           (lambda p: classify_intent(p) == 'pricing', PRICING_PROMPT),
#           DEFAULT_PROMPT,
#       )
#       | llm | StrOutputParser()
#   )


# Drive the howto and pricing fixtures through the routed chain inside
# mlflow.start_run() so the prompt-span content lands in a trace.
TASK4_RUN_IDS = {}
for (qid, q, intent, _kw) in TEST_QUERIES:
    if intent not in ("howto", "pricing"):
        continue
    with mlflow.start_run(run_name=f"task4_q{qid}_{intent}") as run:
        response = routed_chain.invoke(q)
        TASK4_RUN_IDS[intent] = run.info.run_id
        print(f"Q{qid} ({intent}) routed_chain response: {response[:200]!r}")

print(f"\nCaptured run IDs by intent: {TASK4_RUN_IDS}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: invoke routed_chain on the howto fixture and the pricing
# fixture independently, fetch each run's trace, walk the chat-model span's
# input messages (the rendered prompt), and assert the branch-specific
# substring appears (step-by-step for howto, cost for pricing).


def _prompt_content_from_trace(trace):
    """Walk the trace spans and return concatenated input text from any
    chat-model / LLM span. The autolog chat-model span carries the final
    rendered prompt in its `inputs` attribute under a 'messages' key."""
    if trace is None or not getattr(trace, "data", None):
        return ""
    chunks = []
    for span in (trace.data.spans or []):
        span_type = (getattr(span, "span_type", "") or "").upper()
        name = (getattr(span, "name", "") or "").lower()
        if span_type in ("LLM", "CHAT_MODEL") or "chatdatabricks" in name or "llm" in name:
            inputs = getattr(span, "inputs", None) or {}
            try:
                chunks.append(json.dumps(inputs))
            except Exception:
                chunks.append(str(inputs))
    return "\n".join(chunks)


def checkpoint_4(session):
    try:
        chain = globals().get("routed_chain")
        if chain is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Module-level variable `routed_chain` is not defined. Compose "
                    "the RunnableBranch and assign the chain to `routed_chain` before re-running."
                ),
            )
        fixtures = run_sql(
            f"SELECT query, intent_label, expected_keyword FROM {TEST_QUERIES_FQN} "
            f"WHERE intent_label IN ('howto','pricing')"
        )
        if len(fixtures["rows"]) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{TEST_QUERIES_FQN} must contain one 'howto' and one 'pricing' "
                    f"intent_label row. Re-run Task 1 to seed the full TEST_QUERIES fixture."
                ),
            )
        client = MlflowClient()
        for row in fixtures["rows"]:
            q = str(row[0])
            intent = str(row[1])
            expected = str(row[2]).lower()
            with mlflow.start_run(run_name=f"checkpoint4_{intent}") as run:
                response = chain.invoke(q)
            if not isinstance(response, str) or not response.strip():
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"routed_chain.invoke for intent={intent!r} returned empty. "
                        f"Confirm the chain ends in StrOutputParser()."
                    ),
                )
            trace = mlflow.get_trace(run.info.run_id)
            prompt_text = _prompt_content_from_trace(trace).lower()
            if expected and expected not in prompt_text:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Trace for intent={intent!r} did not show branch-specific text "
                        f"{expected!r} in the prompt span. RunnableBranch did not route the "
                        f"query through the {intent} template. Inspect classify_intent() "
                        f"and the branch order in RunnableBranch."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the checkpoint fails on `routed_chain` undefined, the chain has not been composed yet. If it fails on the prompt-span text, the router fired the wrong branch (a 'how much does X cost' query went through HOWTO instead of PRICING because the howto rule matched first on the leading 'how'). Branch order in `RunnableBranch` is first-match-wins; the pricing rule must check before the howto rule, or the howto rule must be stricter than just 'how'.

</details>

<details><summary>Hint 2 (direction)</summary>

`RunnableBranch` takes tuples of `(predicate, runnable)` plus a final default. Predicates receive the upstream payload, so for the parallel-dict head the predicate signature is `lambda payload: bool`. The payload is a dict with keys `context` and `question`. Read `payload['question']`. The pricing branch must be either listed first or its predicate must be stricter than the howto predicate, otherwise 'how much does X cost' matches the howto rule on its leading 'how'.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
def classify_intent(payload):
    q = (payload.get("question") or "").lower().strip()
    if q.startswith("how much") or "cost" in q or "price" in q:
        return "pricing"
    if q.startswith("how do") or q.startswith("how can"):
        return "howto"
    return "default"


routed_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableBranch(
        (lambda p: classify_intent(p) == "pricing", PRICING_PROMPT),
        (lambda p: classify_intent(p) == "howto", HOWTO_PROMPT),
        DEFAULT_PROMPT,
    )
    | llm
    | StrOutputParser()
)
```

</details>

**Wallet check.** Two routed invocations in the task cell plus two more inside the checkpoint, four chain runs total: roughly 2 cents of LLM tokens. Cumulative session spend lands under 5 cents. The MLflow trace UI now has four task2 runs, one task3 set, and four task4 runs, all on the same experiment.

## Cleanup

Time to tear it all down. The cell below runs the three-phase cleanup against `CLEANUP_MANIFEST` in reverse-creation order: the MLflow experiment soft-deletes (Databricks-hosted MLflow moves it to trash; auto-purge handles the rest), then `chain_runs`, then `test_queries`, then the schema with `CASCADE`. The Lab 5 endpoint and index are NOT touched; they survive into Lab 7 through 9.

Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. No critical (hourly-billed)
# resources in this lab; MLflow experiment, two tables, and one schema.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: under $0.05.** Three task-loop chain invocations plus four checkpoint invocations plus two task-4 routed invocations plus four checkpoint-4 routed invocations: roughly a dozen LLM completions on a pay-per-token endpoint at ~600 tokens each. The MLflow trace storage is free on Databricks-hosted MLflow. The Lab 5 endpoint and index are still alive (this lab does not touch them); Lab 9 cleanup is the canonical place to retire them. Your morning coffee cost roughly 80x more than this Databricks session.

## Reflection

These are not graded. They are for you.

1. Walk through what the LCEL `retriever | prompt | llm | parser` chain does at invoke time. Name the data type that flows out of each component into the next, and where the pipe operator's runtime type-coercion would fail if you composed the chain wrong.

2. The intent-classification router branches the prompt template. What is the failure mode when the router itself is wrong (a pricing question routed through the howto template)? How would you detect that failure in production?

## Exam-Style Practice

**Question 1 (Domain 3, prompt augmentation by intent):**

A GenAI engineer is augmenting prompts for a customer-support agent. Customer queries arrive in three flavors: account-balance lookups, shipping ETA requests, and product-spec questions. Each flavor needs a different system prompt and different retrieved context. Which approach is the most maintainable?

A. One monolithic prompt template that handles all three flavors via if/else inside the prompt string.
B. A router (LangChain RunnableBranch or equivalent) that classifies intent and selects one of three prompt templates, each tuned for its flavor.
C. Three separate LLMs, one per flavor, with the engineer manually selecting which LLM to call based on the URL of the inbound request.
D. A single prompt template with the customer's raw query and no augmentation; let the LLM figure out the flavor.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: monolithic prompts with conditional logic inside the prompt string are hard to test and hard to evolve; each flavor's evolution affects the others.
- B is correct: a router that classifies intent and selects a flavor-specific prompt is the LangChain RunnableBranch pattern. Each branch is independently testable and evolvable. The trace shows which branch fired for each query, supporting debugging.
- C is wrong: three separate LLMs is over-engineered; the same LLM with different prompts is sufficient. Tying flavor selection to the URL is a presentation-layer detail leaking into the model layer.
- D is wrong: relying on the LLM to figure out the flavor is brittle; explicit routing produces predictable behavior.

</details>

**Question 2 (Domain 4, chain composition):**

A GenAI engineer is composing a RAG chain with LCEL. The retriever returns a list of LangChain Document objects. The prompt template expects a single string with a `{context}` placeholder. Which step is required between the retriever and the prompt?

A. Nothing; LCEL automatically coerces a Document list into a string.
B. A `RunnableLambda(lambda docs: "\n\n".join(d.page_content for d in docs))` (or equivalent) that joins the documents into a single string.
C. A second retriever that re-retrieves against the first retriever's output.
D. A `StrOutputParser` between the retriever and the prompt.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: LCEL does NOT auto-coerce. The pipe operator forwards the upstream type to the downstream component; a List[Document] passed to a PromptTemplate fails with a type error.
- B is correct: the canonical "join documents into context string" step is a RunnableLambda that maps the Document list to a single string with the desired separator. The string is then interpolated into the prompt's `{context}` placeholder.
- C is wrong: a second retriever does not make sense; the question is about format coercion, not retrieval.
- D is wrong: StrOutputParser parses LLM output to a string; it does not take Document input. Its place is at the end of the chain, not before the prompt.

</details>